# 03 — Preprocessing and split validation

This notebook validates the outputs generated by `scripts/prepare_dataset.py`. It imports the same parser, validation, split, and overlay helpers used by the CLI. It does not implement normalization itself.

Before running, regenerate local manifests:

```bash
python scripts/prepare_dataset.py --dataset-root data/mcocr --output-dir data/processed --seed 42
```

The notebook displays local receipt images and transcript-derived aggregates. Clear outputs before committing.

In [ ]:
from collections import Counter, defaultdict
import json
import os
import random
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

from docucheck.data.loader import REPOSITORY_ROOT, load_kie_records, resolve_dataset_root
from docucheck.data.splitting import find_cross_split_duplicate_groups
from docucheck.data.validation import validate_manifest
from docucheck.visualization.receipt import plot_receipt_record, plot_standardized_record

DATASET_ROOT = resolve_dataset_root(os.environ.get("MC_OCR_ROOT"))
PROCESSED_DIR = REPOSITORY_ROOT / "data" / "processed"
if not PROCESSED_DIR.is_dir():
    raise FileNotFoundError("Run scripts/prepare_dataset.py first to generate data/processed")

def read_jsonl(path):
    with path.open(encoding="utf-8") as stream:
        return [json.loads(line) for line in stream if line.strip()]

manifest = read_jsonl(PROCESSED_DIR / "manifest.jsonl")
recognition_manifest = read_jsonl(PROCESSED_DIR / "recognition_manifest.jsonl")
report = json.loads((PROCESSED_DIR / "validation_report.json").read_text(encoding="utf-8"))
raw_records = load_kie_records(DATASET_ROOT)
manifest_check = validate_manifest(manifest + recognition_manifest)
print("Manifest records:", len(manifest), "| crop recognition records:", len(recognition_manifest))
print("Schema validation:", manifest_check["valid"], manifest_check["issue_counts"])

## Source annotations versus standardized records

A fixed-seed sample from the internal development subset is shown side by side. The left panel uses raw source polygons; the right uses the standardized manifest polygons and canonical field names. Text is hidden in both panels.

In [ ]:
raw_by_name = {record["image_id"]: record for record in raw_records}
processed_by_name = {record["source_image_name"]: record for record in manifest
                     if record["source_split"] == "train" and record["split"] == "internal_dev"}
raw_by_filename = {record["image_id"]: record for record in raw_records}
candidates = [record for record in manifest
              if record["split"] == "internal_dev" and record["regions"]
              and not record["validation_issues"]]
sampled = random.Random(42).sample(candidates, min(2, len(candidates)))

for processed in sampled:
    source = raw_by_filename[processed["source_image_name"]]
    fig, axes = plt.subplots(1, 2, figsize=(14, 7))
    plot_receipt_record(source, ax=axes[0], show_text=False)
    axes[0].set_title("Source KIE polygons")
    plot_standardized_record(processed, dataset_root=DATASET_ROOT, ax=axes[1], show_text=False)
    plt.suptitle("Visual alignment check: raw versus processed")
    plt.tight_layout()
    plt.show()

# Compare source points and derived bounds numerically for one sample without displaying text.
coordinate_rows = []
for processed in sampled[:1]:
    source = raw_by_filename[processed["source_image_name"]]
    for index, (raw_region, clean_region) in enumerate(zip(source["regions"], processed["regions"])):
        coordinate_rows.append({
            "region_index": index,
            "label_raw": clean_region["label_raw"],
            "raw_first_point": raw_region["polygon"][0] if raw_region["polygon"] else None,
            "processed_first_point": clean_region["polygon"][0] if clean_region["polygon"] else None,
            "derived_bbox_xyxy": clean_region["bbox_xyxy"],
            "validation_issues": clean_region["validation_issues"],
        })
display(pd.DataFrame(coordinate_rows).head(8))

## Label mapping and schema diagnostics

Source labels are retained verbatim, canonical fields are populated only for verified mappings, and other labels remain unmapped. Missing annotations are represented as statuses rather than guessed values.

In [ ]:
mapping_counts = Counter()
for record in manifest:
    for region in record.get("regions", []):
        mapping_counts[(region.get("label_raw"), region.get("canonical_field"))] += 1
mapping_table = pd.DataFrame([
    {"source_label": raw, "canonical_field": canonical or "unmapped", "regions": count}
    for (raw, canonical), count in sorted(mapping_counts.items(), key=lambda item: (str(item[0][0]), str(item[0][1])))
])
display(mapping_table)

status_counts = Counter(record.get("annotation_status") for record in manifest)
image_status_counts = Counter(record.get("image_status") for record in manifest)
geometry_by_source = Counter()
for record in manifest:
    for region in record.get("regions", []) + record.get("ocr_regions", []):
        for issue in region.get("validation_issues", []):
            geometry_by_source[(region.get("annotation_source"), issue)] += 1
print("Annotation statuses:", dict(status_counts))
print("Image statuses:", dict(image_status_counts))
print("Invalid/skipped record counts:", report["invalid_record_count"], report["skipped_record_count"])
print("Geometry issues by source:", {f"{source}: {issue}": count for (source, issue), count in geometry_by_source.items()})

## Split integrity and duplicates

The official validation image inventory remains separate from the team-created internal development split. Exact SHA-256 duplicates are grouped before splitting. Groups matching official validation are quarantined from internal training and development. Source/template metadata was not available in the observed files, so no template grouping is claimed.

In [ ]:
split_counts = pd.Series(Counter(record["split"] for record in manifest), name="receipt records")
recognition_split_counts = pd.Series(Counter(record["split"] for record in recognition_manifest), name="crop records")
display(pd.concat([split_counts, recognition_split_counts], axis=1).fillna(0).astype(int))

exact_groups = defaultdict(list)
for record in manifest:
    if record.get("image_group_id"):
        exact_groups[record["image_group_id"]].append(record)
duplicate_group_sizes = [len(group) for group in exact_groups.values() if len(group) > 1]
cross_split_duplicates = find_cross_split_duplicate_groups(manifest + recognition_manifest)
print("Exact duplicate groups / records in groups:", len(duplicate_group_sizes), sum(duplicate_group_sizes))
print("Cross-split duplicate group count:", len(cross_split_duplicates))
print("Duplicate groups quarantined from official validation:", report["official_validation_duplicate_quarantine_groups"])
print("CLI split counts match manifest:", dict(Counter(record["split"] for record in manifest)) == report["records_by_split"])

# Private inspection hook: call with a flagged hash locally if a future run reports a crossover.
def inspect_duplicate_group_locally(group_hash):
    return [
        {"image_id": record["image_id"], "source_image_name": record.get("source_image_name"),
         "split": record.get("split"), "image_path": record["image_path"]}
        for record in manifest + recognition_manifest
        if record.get("image_group_id") == group_hash
    ]
print("Manifest structural validation:", manifest_check["valid"])

## Preprocessing conclusions

- The CLI preserves raw text, source labels, source geometry, category IDs, source split, and annotation provenance; it adds opaque IDs, dimensions, exact-file hashes, normalized field names, and `bbox_xyxy` only for valid geometry.
- The four expected KIE labels map from `SELLER`, `ADDRESS`, `TIMESTAMP`, and `TOTAL_COST`. The single `TOTAL_TOTAL_COST` region remains unmapped.
- The internal split is deterministic with seed 42. Exact duplicates do not cross internal train/development; training images duplicated in official validation are assigned to an excluded quarantine split.
- There are 425 out-of-image full-page transcript polygons across 109 images. They remain present with `bbox_xyxy=null` and a validation issue. The default `report` policy keeps them visible; `skip` and `fail` are available for stricter runs.
- The 391 official validation images have no KIE label/polygon columns in this download, so they are preserved as inventory records and are not used for field-level metrics.

**Readiness:** standardized records are structurally valid and reproducible. KIE geometry is usable in the observed train set; full-page OCR annotations require filtering or explicit invalid-geometry handling before quantitative detection evaluation.